In [ ]:
!pip install transformers datasets evaluate accelerate -q

import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
import evaluate

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00
PyTorch version: 2.10.0+cu128
GPU available: True
Device: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/automotive-nlp'

df = pd.read_csv(os.path.join(DATA_DIR, 'complaints_clean.csv'))

with open(os.path.join(DATA_DIR, 'label_mapping.json'), 'r') as f:
    mapping = json.load(f)

label2id = mapping['label2id']
id2label = mapping['id2label']

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df['label'].value_counts())
print("\nLabel mapping:")
print(label2id)
print("\nSample:")
print(df[['clean_complaint', 'label', 'label_id']].head(3))

Mounted at /content/drive
Dataset shape: (18000, 3)

Class distribution:
label
ENGINE        3000
POWERTRAIN    3000
AIRBAGS       3000
STEERING      3000
ELECTRICAL    3000
BRAKES        3000
Name: count, dtype: int64

Label mapping:
{'AIRBAGS': 0, 'BRAKES': 1, 'ELECTRICAL': 2, 'ENGINE': 3, 'POWERTRAIN': 4, 'STEERING': 5}

Sample:
                                     clean_complaint   label  label_id
0  consumer states that while driving at any spee...  ENGINE         3
1  2001 ford excursion 5.4l triton v8. at 89,000 ...  ENGINE         3
2  2006 nissan pathfinder in august the fuel gaug...  ENGINE         3


## Train / Validation / Test Split

We split the data into three sets:
- Train (70%): used to fine-tune DistilBERT weights
- Validation (15%): used during training to monitor progress and prevent overfitting
- Test (15%): held out completely, only used for final evaluation

stratify=y ensures each split has the same class proportions.
This is critical — without it you could end up with a test set
that has very few samples of one class, making evaluation unreliable.

In [ ]:
X = df['clean_complaint'].values
y = df['label_id'].values

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp
)

print(f"Train size:      {len(X_train):,}")
print(f"Validation size: {len(X_val):,}")
print(f"Test size:       {len(X_test):,}")
print(f"Total:           {len(X_train) + len(X_val) + len(X_test):,}")

Train size:      12,607
Validation size: 2,693
Test size:       2,700
Total:           18,000


## Tokenization

DistilBERT does not read raw text — it reads token IDs.
The tokenizer converts text to IDs through these steps:
1. Split text into subword tokens (e.g. "overheating" → ["over", "##heat", "##ing"])
2. Add special tokens: [CLS] at start, [SEP] at end
3. Convert tokens to integer IDs from the vocabulary
4. Pad or truncate to max_length=128

Why max_length=128 instead of 512?
Most complaints are under 128 tokens. Using 512 would quadruple
training time for no meaningful accuracy gain. 128 is the
standard choice for short text classification tasks.

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(texts, max_length=128):
    return tokenizer(
        list(texts),
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )

print("Tokenizing train set...")
train_encodings = tokenize(X_train)

print("Tokenizing validation set...")
val_encodings = tokenize(X_val)

print("Tokenizing test set...")
test_encodings = tokenize(X_test)

print("Tokenization complete.")
print(f"Train input_ids shape: {train_encodings['input_ids'].shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing train set...
Tokenizing validation set...
Tokenizing test set...
Tokenization complete.
Train input_ids shape: torch.Size([12607, 128])


## PyTorch Dataset class

The HuggingFace Trainer expects data in a specific format —
a PyTorch Dataset object that returns one sample at a time.

Each sample is a dictionary with:
- input_ids: token ID sequence (length 128)
- attention_mask: 1 for real tokens, 0 for padding
- labels: integer class label (0-5)

The attention_mask tells DistilBERT which tokens are real
and which are just padding — it ignores padding positions.

In [ ]:
class ComplaintDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: val[idx] for key, val in self.encodings.items()
        }
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = ComplaintDataset(train_encodings, y_train)
val_dataset = ComplaintDataset(val_encodings, y_val)
test_dataset = ComplaintDataset(test_encodings, y_test)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset:   {len(val_dataset)} samples")
print(f"Test dataset:  {len(test_dataset)} samples")

sample = train_dataset[0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"input_ids shape: {sample['input_ids'].shape}")
print(f"label: {sample['labels']}")

Train dataset: 12607 samples
Val dataset:   2693 samples
Test dataset:  2700 samples

Sample keys: ['input_ids', 'attention_mask', 'labels']
input_ids shape: torch.Size([128])
label: 2


## Loading DistilBERT for sequence classification

DistilBertForSequenceClassification loads the pretrained DistilBERT
weights and adds a classification head on top — a single linear layer
that maps the [CLS] token representation to num_labels output scores.

The model has never seen our specific labels before.
Fine-tuning adjusts all the weights slightly to perform well
on our complaint classification task.

num_labels=6 tells it we have 6 output classes.
id2label and label2id allow the model to display readable
label names when making predictions.

In [ ]:
num_labels = len(label2id)

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels,
    id2label={int(k): v for k, v in id2label.items()},
    label2id=label2id
)

print("Model loaded.")
print(f"Number of labels: {num_labels}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.
Number of labels: 6
Model parameters: 66,958,086


## Training arguments

These hyperparameters control how training runs.

Key decisions:
- num_train_epochs=4: 4 full passes through the training data.
  More epochs = better fit but risk of overfitting. 4 is standard.
- per_device_train_batch_size=32: 32 samples per GPU step.
  Larger = faster but needs more GPU memory. 32 fits comfortably on T4.
- learning_rate=2e-5: standard fine-tuning rate for BERT-family models.
  Too high destroys pretrained weights. Too low = no learning.
- evaluation_strategy='epoch': evaluate on validation set after every epoch.
  Lets you see if the model is improving or overfitting.
- load_best_model_at_end=True: automatically keeps the best checkpoint.
  If epoch 3 is best but epoch 4 overfits, you get epoch 3's weights.
- warmup_ratio=0.1: gradually increases learning rate for first 10% of steps.
  Prevents unstable updates at the start of training.

In [ ]:
OUTPUT_PATH = '/content/drive/MyDrive/automotive-nlp/model_output'

training_args = TrainingArguments(
    output_dir=OUTPUT_PATH,
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_dir='/content/logs',
    logging_steps=50,
    fp16=True,
    report_to='none'
)

print("Training arguments set.")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments set.
Epochs: 4
Batch size: 32
Learning rate: 2e-05


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    f1 = evaluate.load('f1')
    precision = evaluate.load('precision')
    recall = evaluate.load('recall')

    return {
        'f1': f1.compute(
            predictions=predictions, references=labels, average='macro'
        )['f1'],
        'precision': precision.compute(
            predictions=predictions, references=labels, average='macro'
        )['precision'],
        'recall': recall.compute(
            predictions=predictions, references=labels, average='macro'
        )['recall'],
    }

print("Metrics function defined.")

Metrics function defined.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training...")
print("Expected time: 15-25 minutes on T4 GPU")
print("Watch the loss decrease and F1 increase across epochs.\n")

trainer.train()

print("\nTraining complete.")

Starting training...
Expected time: 15-25 minutes on T4 GPU
Watch the loss decrease and F1 increase across epochs.



Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.713936,0.664348,0.786364,0.787726,0.788396
2,0.608506,0.620385,0.791899,0.791844,0.793218
3,0.503643,0.634071,0.791923,0.793397,0.794330
4,0.427302,0.627074,0.798163,0.797755,0.798780


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Training complete.


In [ ]:
print("Evaluating on test set...")
test_results = trainer.evaluate(test_dataset)

print("\nTest set results:")
for key, val in test_results.items():
    if isinstance(val, float):
        print(f"  {key}: {val:.4f}")

Evaluating on test set...



Test set results:
  eval_loss: 0.6432
  eval_f1: 0.7904
  eval_precision: 0.7905
  eval_recall: 0.7907
  eval_runtime: 3.6702
  eval_samples_per_second: 735.6630
  eval_steps_per_second: 11.7160
  epoch: 4.0000


In [ ]:
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)

label_names = [id2label[str(i)] for i in range(num_labels)]

print("Per-class Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_names))

Per-class Classification Report:
              precision    recall  f1-score   support

     AIRBAGS       0.90      0.90      0.90       450
      BRAKES       0.77      0.80      0.79       450
  ELECTRICAL       0.71      0.68      0.69       450
      ENGINE       0.76      0.80      0.78       450
  POWERTRAIN       0.79      0.75      0.77       450
    STEERING       0.81      0.82      0.82       450

    accuracy                           0.79      2700
   macro avg       0.79      0.79      0.79      2700
weighted avg       0.79      0.79      0.79      2700



In [ ]:
FINAL_MODEL_PATH = '/content/drive/MyDrive/automotive-nlp/final_model'

trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)

print(f"Model saved to: {FINAL_MODEL_PATH}")
print("\nFiles saved:")
for f in os.listdir(FINAL_MODEL_PATH):
    size = os.path.getsize(os.path.join(FINAL_MODEL_PATH, f)) / (1024*1024)
    print(f"  {f} — {size:.1f} MB")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/automotive-nlp/final_model

Files saved:
  model.safetensors — 255.4 MB
  training_args.bin — 0.0 MB
  config.json — 0.0 MB
  tokenizer_config.json — 0.0 MB
  tokenizer.json — 0.7 MB


In [ ]:
results_to_save = {
    'test_f1': test_results.get('eval_f1', 0),
    'test_precision': test_results.get('eval_precision', 0),
    'test_recall': test_results.get('eval_recall', 0),
    'test_loss': test_results.get('eval_loss', 0),
    'num_labels': num_labels,
    'label_names': label_names,
    'model': 'distilbert-base-uncased',
    'epochs': 4,
    'batch_size': 32,
    'max_length': 128
}

with open(os.path.join(DATA_DIR, 'training_results.json'), 'w') as f:
    json.dump(results_to_save, f, indent=2)

print("Results saved to Google Drive.")
print("\nYour numbers for the resume:")
print(f"  F1 Score:  {results_to_save['test_f1']:.4f}")
print(f"  Precision: {results_to_save['test_precision']:.4f}")
print(f"  Recall:    {results_to_save['test_recall']:.4f}")

Results saved to Google Drive.

Your numbers for the resume:
  F1 Score:  0.7904
  Precision: 0.7905
  Recall:    0.7907


In [ ]:
import shutil
from google.colab import files

zip_path = '/content/final_model.zip'
shutil.make_archive('/content/final_model', 'zip', FINAL_MODEL_PATH)

print(f"Zip size: {os.path.getsize(zip_path) / (1024*1024):.1f} MB")
print("Downloading to your computer...")

files.download(zip_path)

print("Download started. Extract and move to automotive-complaint-nlp/models/final_model/")

Zip size: 235.8 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started. Extract and move to automotive-complaint-nlp/models/final_model/


## Training Summary

Model: distilbert-base-uncased (fine-tuned)
Dataset: NHTSA Vehicle Complaints — 6 component categories
Training samples: ~12,600
Validation samples: ~2,700
Test samples: ~2,700

Training configuration:
- Epochs: 4
- Batch size: 32
- Learning rate: 2e-5
- Max sequence length: 128
- Mixed precision (fp16): enabled

Results on held-out test set: see training_results.json

Model saved to:
- Google Drive: automotive-nlp/final_model/
- Local: models/final_model/

Next: Build Streamlit app for real-time classification